In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

df = pd.read_csv("students_feature_engineered_v2_large.csv")

X = df.drop(columns=["Pass"])
y = df["Pass"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train Shape:", X_train.shape)
print("Test Shape :", X_test.shape)


model = LogisticRegression(max_iter=500)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"\nAccuracy (random_state=42): {acc:.4f}")


scores = []

for seed in [0, 7, 13, 21, 99]:

    X_train_seed, X_test_seed, y_train_seed, y_test_seed = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=seed,
        stratify=y
    )

    model = LogisticRegression(max_iter=500)
    model.fit(X_train_seed, y_train_seed)
    y_pred_seed = model.predict(X_test_seed)
    score = accuracy_score(y_test_seed, y_pred_seed)
    scores.append(score)
    print(f"Seed {seed}: Accuracy = {score:.4f}")


score_range = max(scores) - min(scores)
print("\nScores:", scores)
print(f"Score Range = {score_range:.4f}")

print(
    "\nImplication: Different train/test splits can produce "
    "different accuracy values. A large range suggests that "
    "a single split may not provide a reliable estimate of "
    "model performance."
)

X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

print("\nFinal Shapes")
print("Train      :", X_train_final.shape)
print("Validation :", X_val.shape)
print("Test       :", X_test.shape)

Train Shape: (800, 6)
Test Shape : (200, 6)

Accuracy (random_state=42): 1.0000
Seed 0: Accuracy = 1.0000
Seed 7: Accuracy = 1.0000
Seed 13: Accuracy = 1.0000
Seed 21: Accuracy = 1.0000
Seed 99: Accuracy = 1.0000

Scores: [1.0, 1.0, 1.0, 1.0, 1.0]
Score Range = 0.0000

Implication: Different train/test splits can produce different accuracy values. A large range suggests that a single split may not provide a reliable estimate of model performance.

Final Shapes
Train      : (640, 6)
Validation : (160, 6)
Test       : (200, 6)


In [ ]:
# A large score range indicates that model performance depends on how the data is split.
# Therefore, a single train/test split may give a misleading estimate of real-world performance.
# Cross-validation provides a more reliable evaluation because it averages performance across multiple splits.

In [2]:
#task2
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


df = pd.read_csv("students_feature_engineered_v2_large.csv")

X = df.drop(columns=["Pass"])
y = df["Pass"]


kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = []
for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):

    
    X_tr = X.values[train_idx]
    X_val = X.values[val_idx]

    y_tr = y.values[train_idx]
    y_val = y.values[val_idx]

    scaler = StandardScaler()

    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)

    model = LogisticRegression(max_iter=500)
    model.fit(X_tr_scaled, y_tr)

    y_pred = model.predict(X_val_scaled)

    acc = accuracy_score(y_val, y_pred)

    fold_scores.append(acc)

    print(f"Fold {fold}: {acc:.3f}")

print("\nKFold Results")
print("Scores:", fold_scores)
print(f"Mean: {np.mean(fold_scores):.3f}")
print(f"Std : {np.std(fold_scores):.3f}")


print("\n" + "="*40)
print("StratifiedKFold")
print("="*40)

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

strat_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):

    X_tr = X.values[train_idx]
    X_val = X.values[val_idx]

    y_tr = y.values[train_idx]
    y_val = y.values[val_idx]

    scaler = StandardScaler()

    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)

    model = LogisticRegression(max_iter=500)

    model.fit(X_tr_scaled, y_tr)

    y_pred = model.predict(X_val_scaled)

    acc = accuracy_score(y_val, y_pred)

    strat_scores.append(acc)

    print(f"Fold {fold}: {acc:.3f}")

print("\nStratifiedKFold Results")
print("Scores:", strat_scores)
print(f"Mean: {np.mean(strat_scores):.3f}")
print(f"Std : {np.std(strat_scores):.3f}")

Fold 1: 1.000
Fold 2: 1.000
Fold 3: 1.000
Fold 4: 1.000
Fold 5: 1.000

KFold Results
Scores: [1.0, 1.0, 1.0, 1.0, 1.0]
Mean: 1.000
Std : 0.000

StratifiedKFold
Fold 1: 1.000
Fold 2: 1.000
Fold 3: 1.000
Fold 4: 1.000
Fold 5: 1.000

StratifiedKFold Results
Scores: [1.0, 1.0, 1.0, 1.0, 1.0]
Mean: 1.000
Std : 0.000


In [ ]:
# Why fit the scaler inside the fold?

# If the scaler is fitted before cross-validation, information from the validation fold leaks into the training process 
# through the scaling statistics (mean and standard deviation). This is called data leakage.
# By fitting the scaler only on the training portion of each fold, the validation data remains completely unseen.


# When does StratifiedKFold matter more than KFold?

# StratifiedKFold is especially important for imbalanced datasets because it preserves class proportions in every fold,
# leading to more reliable evaluation scores.

In [3]:
#task3
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, cross_validate

df = pd.read_csv("students_feature_engineered_v2_large.csv")

X = df.drop(columns=["Pass"])
y = df["Pass"]

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=500))
])

acc_scores = cross_val_score(
    pipe,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("Accuracy Scores:")
print(acc_scores)

print(f"Mean Accuracy: {acc_scores.mean():.3f}")
print(f"Std Accuracy : {acc_scores.std():.3f}")

f1_scores = cross_val_score(
    pipe,
    X,
    y,
    cv=5,
    scoring="f1"
)

print("\nF1 Scores:")
print(f1_scores)

print(f"Mean F1 Score: {f1_scores.mean():.3f}")


cv = cross_validate(
    pipe,
    X,
    y,
    cv=5,
    scoring=["accuracy", "f1"],
    return_train_score=True
)

print("\nCross Validate Output:")
print(cv)

print("\nAccuracy from cross_val_score:")
print(acc_scores)

print("\nAccuracy from cross_validate:")
print(cv["test_accuracy"])

print(
    "\nDo they match?",
    np.allclose(acc_scores, cv["test_accuracy"])
)

print("\nTrain Accuracy:")
print(cv["train_accuracy"])

print("\nTest Accuracy:")
print(cv["test_accuracy"])

Accuracy Scores:
[1. 1. 1. 1. 1.]
Mean Accuracy: 1.000
Std Accuracy : 0.000

F1 Scores:
[1. 1. 1. 1. 1.]
Mean F1 Score: 1.000

Cross Validate Output:
{'fit_time': array([0.01249599, 0.00953197, 0.01097012, 0.01191401, 0.01076627]), 'score_time': array([0.00776792, 0.02359438, 0.01405263, 0.01151109, 0.01460528]), 'test_accuracy': array([1., 1., 1., 1., 1.]), 'train_accuracy': array([1., 1., 1., 1., 1.]), 'test_f1': array([1., 1., 1., 1., 1.]), 'train_f1': array([1., 1., 1., 1., 1.])}

Accuracy from cross_val_score:
[1. 1. 1. 1. 1.]

Accuracy from cross_validate:
[1. 1. 1. 1. 1.]

Do they match? True

Train Accuracy:
[1. 1. 1. 1. 1.]

Test Accuracy:
[1. 1. 1. 1. 1.]


In [ ]:
# When is F1 a better choice than accuracy?

# F1 score is better when the dataset is imbalanced or when false positives and false negatives are important.
# It balances precision and recall, whereas accuracy can be misleading if one class dominates.
#also precision and recall is a trade off


# Do the accuracy arrays from steps 2 and 4 match? Why?

# Yes. Both functions use the same model, dataset, cross-validation folds, and scoring metric. cross_val_score() 
# returns only the test scores, while cross_validate() returns additional information such as train scores and 
# multiple metrics. Therefore, cv["test_accuracy"] should match the accuracy scores from cross_val_score().

In [ ]:
#task 4
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_validate

df = pd.read_csv("students_feature_engineered_v2_large.csv")

X = df.drop(columns=["Pass"])
y = df["Pass"]


models = {
    "LogisticRegression": LogisticRegression(max_iter=500),
    "DecisionTree_full": DecisionTreeClassifier(max_depth=None, random_state=42),
    "DecisionTree_depth3": DecisionTreeClassifier(max_depth=3, random_state=42),
}

rows = []
-

for name, clf in models.items():

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", clf)
    ])

    cv = cross_validate(
        pipe,
        X,
        y,
        cv=5,
        scoring="accuracy",
        return_train_score=True
    )

    mean_train = cv["train_score"].mean()
    mean_test = cv["test_score"].mean()
    gap = mean_train - mean_test

    rows.append({
        "Model": name,
        "Mean_Train": mean_train,
        "Mean_Test": mean_test,
        "Gap": gap
    })


results = pd.DataFrame(rows)

print(results.round(3))

                 Model  Mean_Train  Mean_Test  Gap
0   LogisticRegression         1.0        1.0  0.0
1    DecisionTree_full         1.0        1.0  0.0
2  DecisionTree_depth3         1.0        1.0  0.0


In [3]:
#task5
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv("students_feature_engineered_v2_large.csv")

X = df.drop(columns=["Pass"])
y = df["Pass"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", DecisionTreeClassifier(random_state=42))
])


param_grid = {
    "clf__max_depth": [2, 3, 4, 5, None],
    "clf__min_samples_split": [2, 5, 10],
    "clf__criterion": ["gini", "entropy"]
}

combinations = (
    len(param_grid["clf__max_depth"]) *
    len(param_grid["clf__min_samples_split"]) *
    len(param_grid["clf__criterion"])
)

print(f"Total combinations = {combinations}")
print(f"Total model fits = {combinations * 5}")

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    refit=True
)

grid.fit(X_train, y_train)

print("\nBest Parameters:")
print(grid.best_params_)

print("\nBest CV Accuracy:")
print(grid.best_score_)

results = pd.DataFrame(grid.cv_results_)

cols = [
    "param_clf__max_depth",
    "param_clf__min_samples_split",
    "param_clf__criterion",
    "mean_test_score",
    "std_test_score",
    "rank_test_score"
]

print("\nTop 5 Models:")
print(
    results[cols]
    .sort_values("rank_test_score")
    .head(5)
)

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

print("\nTest Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Total combinations = 30
Total model fits = 150

Best Parameters:
{'clf__criterion': 'gini', 'clf__max_depth': 2, 'clf__min_samples_split': 2}

Best CV Accuracy:
1.0

Top 5 Models:
  param_clf__max_depth  param_clf__min_samples_split param_clf__criterion  \
0                    2                             2                 gini   
1                    2                             5                 gini   
2                    2                            10                 gini   
3                    3                             2                 gini   
4                    3                             5                 gini   

   mean_test_score  std_test_score  rank_test_score  
0              1.0             0.0                1  
1              1.0             0.0                1  
2              1.0             0.0                1  
3              1.0             0.0                1  
4              1.0             0.0                1  

Test Accuracy:
1.0

Classificati

In [ ]:
#stratified 
#train and test